# HIPPIE: Hierarchical Identification of Putative Populations In Electrophysiology

A **trimodal Conditional Variational Autoencoder (CVAE)** for automatic neuron classification from high-density MEA and Neuropixels recordings.

---

## What does HIPPIE do?

Given a recorded neuron, HIPPIE takes three electrophysiological signatures:
- **Waveform** — the spike shape (50 time points after resampling)
- **ISI distribution** — interspike interval histogram (100 bins, log-transformed)
- **ACG** — autocorrelogram (100 bins)

...and learns a low-dimensional latent representation `z` where neurons of the same type cluster together. This enables:
1. **Classification** — KNN or MLP on `z` to predict cell type
2. **Generalization** — same `z` works across datasets and recording technologies  
3. **Imputation** — reconstruct missing modalities from the others

## 3-Phase Training Pipeline

```
Phase 1: Pretrain  (unsupervised, pooled data — no labels needed)
          ↓ transfer encoder weights
Phase 2: Finetune  (unsupervised, target dataset — no labels needed)
          ↓ transfer weights
Phase 3: Supervised  (labeled subset, class embedding active in decoder only)
          ↓
Evaluation: KNN probe on frozen encoder z (class labels never passed to encoder)
```


---
## Architecture

```
                    ENCODER  (class-agnostic at all times)
┌──────────┐  ┌─────────────┐  ┌───────────┐  ┌─────────────┐  ┌──────┐  ┌─────────────┐
│ Waveform │─▶│ ResNet18Enc │  │ ISI dist  │─▶│ ResNet18Enc │  │ ACG  │─▶│ ResNet18Enc │
└──────────┘  └──────┬──────┘  └───────────┘  └──────┬──────┘  └──────┘  └──────┬──────┘
                      └──────────────────┬────────────────────────────────────────┘
                                         │ concat(h_wave, h_isi, h_acg)
                           ┌─────────────▼──────────────┐
                           │  Fusion FC + BatchNorm     │ ← source_emb  (always on)
                           │                            │   super_region_emb (optional)
                           │  [class_emb NEVER here]    │   layer_emb   (optional)
                           └─────────────┬──────────────┘
                                         │
                                 ┌───────▼───────┐
                                 │  z_mean, σ²   │  ← VAE reparameterization trick
                                 └───────┬───────┘
                                         │ z  (the latent representation)
                                         ▼
                    DECODER  (class-conditioned during training only)
                    ┌──────────────────────────────────────────┐
                    │  z + source_emb + class_emb (train only) │
                    │      + super_region_emb + layer_emb      │
                    └──────────────┬───────────────────────────┘
                                   │
                    ┌──────────────▼───────────────────────────┐
                    │  Decoder FC → ResNet18Dec × 3 modalities │
                    └──────────────────────────────────────────┘
```

**The key architectural insight:** class labels feed only the decoder during training. The encoder is class-agnostic by construction. This prevents *spatial drift* (explained below).


---
## Deep Dive: The Four Types of Conditioning

HIPPIE supports four embedding types. Each one answers a different question and has a different availability at inference time.

---
### 1. Source Embedding  —  *"Which recording technology?"*

**What:** A learned 5-dim vector per dataset/technology (e.g., Neuropixels, tetrode, MEA). Each dataset gets a unique integer ID.

**Where:** Both encoder AND decoder — it is *always* available at inference time since you always know which equipment was used.

**Why it helps:** Different recording setups produce systematically shifted waveform shapes and noise floors for the same cell type. Source embedding lets the model factor out technology-specific biases before clustering.

```python
# Each dataset gets a unique integer:
AVAILABLE_DATASETS = {
    "hausser_cell_type":            source_id=1,   # juxtacellular
    "hull_cell_type":               source_id=2,   # MEA
    "lissberger_labeled_cell_type": source_id=3,   # tetrode
    "allen_scope_neuropixel_area":  source_id=4,   # Neuropixels
}
```

---
### 2. Class Embedding  —  *"What cell type is this?"*

**What:** A learned 5-dim vector per cell type (PV, SST, Purkinje, GoC…). Only available during supervised training — you don't know the label at test time.

**The critical design question:** Does the class embedding go into the encoder, the decoder, or both?

#### ❌ Option A — Symmetric conditioning (encoder + decoder): DO NOT USE
```
Train: Input + class_label → Encoder → z → Decoder + class_label → Reconstruction
Test:  Input + ???          → Encoder → z (encoder expects a label signal that isn't there!)
```
At test time you mask class labels. But the encoder was trained expecting a class signal — when you zero it out, all cells collapse to a single cluster in latent space (*spatial drift*). This was the major bug fixed for the Nature Communications rebuttal. Rung 3 (`with_both_embeddings`) in the ladder shows ~0.47 accuracy due to this failure mode.

#### ✅ Option B — Asymmetric conditioning (decoder only): PRODUCTION
```
Train: Input → Encoder → z → Decoder + class_label → Reconstruction
Test:  Input → Encoder → z  ← exact same encoder path, no mismatch!
```
The encoder never sees class labels — not during training, not at test time. The decoder uses them during training to condition reconstruction, which teaches `z` to encode class-discriminative features. Evaluation path is perfectly clean.

Controlled by one flag: `encoder_uses_class_embedding=False` (set in all "rung 4+" configs).

#### Three decoder-side regularizers (prevent decoder over-reliance from leaking back):

| Regularizer | Mechanism | Value |
|-------------|-----------|-------|
| **Class dropout** | Randomly zero class embedding during training | p=0.3 |
| **Consistency loss** | MSE between decoded-with-class and decoded-without-class | λ=0.15 |
| **Warmup schedule** | Scale consistency loss 0→1 over first N epochs | 5 epochs |

---
### 3. Super Region Embedding  —  *"Which brain region?"*

**What:** A learned embedding for coarse brain region (hippocampus, cortex, cerebellum, striatum). Available at both train AND test time — you always know where you recorded from.

**Where:** Both encoder and decoder (like source embedding).

**Why it helps:** The same inhibitory interneuron looks different in hippocampus vs cortex. Super region embedding lets the model factor out region-level baseline variation without treating it as a class label.

```python
# Enable with:
config.use_super_region_embedding = True
model = MultiModalCVAE(..., num_super_regions=5)
model(data, source_labels=src, super_region_labels=region)
```

---
### 4. Layer Embedding  —  *"Which cortical or cerebellar layer?"*

**What:** A learned embedding for laminar position (L2/3, L4, L5, L6 in cortex; PCL, GCL, ML in cerebellum). Available at both train AND test time via histology / probe depth estimation.

**Where:** Both encoder and decoder.

**Why it helps:** Within a brain region, cells in different layers have distinct electrophysiological baselines. Layer embedding removes this within-region confound and focuses `z` on cell-type-specific features.

```python
# Enable with:
config.use_layer_embedding = True
model = MultiModalCVAE(..., num_layers=8)
model(data, source_labels=src, layer_labels=layer)
```


---
## The Conditioning Ladder: 8 Rungs from Baseline to Production

The Nature Communications rebuttal systematically benchmarked 8 configurations. Each rung adds exactly one component:

| Rung | Config name | Enc sees class? | Source | Class | BN | Aug | Reg | Notes |
|------|-------------|----------------|--------|-------|----|-----|-----|-------|
| 0 | `baseline` | — | ❌ | ❌ | ❌ | ❌ | ❌ | Pure VAE |
| 1 | `with_source` | — | ✅ | ❌ | ❌ | ❌ | ❌ | + tech embedding |
| 2 | `with_class` | ✅ leaky | ❌ | ✅ | ❌ | ❌ | ❌ | + class (sym, encoder drifts) |
| 3 | `with_both_embeddings` | ✅ leaky | ✅ | ✅ | ❌ | ❌ | ❌ | Both (leaky — spatial drift!) |
| 4 | `class_decoder_source` | **❌ fixed** | ✅ | ✅ | ❌ | ❌ | ❌ | **THE KEY FIX: decoder-only class** |
| 5 | `class_decoder_source_bn` | ❌ | ✅ | ✅ | ✅ | ❌ | ❌ | + batch norm |
| 6 | `class_decoder_source_bn_strong_aug` | ❌ | ✅ | ✅ | ✅ | Strong | ❌ | Strong aug |
| 7 | **`class_decoder_source_bn_aug_reg`** | ❌ | ✅ | ✅ | ✅ | Light | ✅ | **PRODUCTION** |

**The biggest jump:** Rung 3→4: moving class from encoder to decoder gives ~+0.13 mean balanced accuracy.

**Why Rung 7 over Rung 5?** Hausser dataset (2nd most representative of the 11-dataset benchmark) scores +0.069 higher with Rung 7. Hausser is closer to 9/9 other datasets than CellExplorer is, so its advantage is more likely to generalize.

**Also available:** `HIPPIE_contrastive` — Rung 7 + supervised contrastive loss on `z_mean`. Adds explicit discriminative pressure by pushing same-class embeddings together and different-class embeddings apart. Particularly useful for datasets with many confusable cell types.


---
## 1. Setup and Imports

In [ ]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Add hippie package to path
code_dir = os.path.abspath(os.path.join(os.getcwd(), 'hippie'))
sys.path.insert(0, code_dir)

import torch
import torch.nn as nn
import pytorch_lightning as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score, train_test_split
from torch.utils.data import random_split, WeightedRandomSampler

from multimodal_model import MultiModalCVAE, MultiModalCVAETrainModule, CVAEConfig, ExperimentConfigs
from dataloading import MultiModalEphysDataset, none_safe_collate
from augmentations import AugmentedMultiModalEphysDataset

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")
print(f"\nAvailable HIPPIE configs:")
for name in sorted(m for m in dir(ExperimentConfigs) if not m.startswith('_')):
    print(f"  • {name}")

---
## 2. Configuration

Edit this cell to set up your experiment. All parameters are documented.

In [ ]:
# ─── Dataset selection ────────────────────────────────────────────────────────
# TRAINING_DATASET: must have cell type labels (used for supervised training and KNN fitting)
# PREDICT_DATASET:  can be labeled (to evaluate accuracy) or unlabeled (produces predictions only)

TRAINING_DATASET = "hausser_cell_type"
PREDICT_DATASET  = "lissberger_labeled_cell_type"

# All datasets and their source IDs (technology-identifier for source embedding)
# Add any dataset folder from datasets_hippie/ here.
AVAILABLE_DATASETS = {
    "hausser_cell_type":              1,   # juxtacellular, cerebellar
    "hull_cell_type":                 2,   # MEA, cerebellar
    "lissberger_labeled_cell_type":   3,   # tetrode, cerebellar
    "cellexplorer_cell_type":         4,   # Neuropixels, cortical
    "allen_scope_neuropixel_area":    5,   # Neuropixels, multi-region
    "mouse_organoids_cell_line":      6,   # MEA, organoid
    "juxtacellular_mouse_s1_area":    7,   # juxtacellular, S1
}

# ─── Model configuration ──────────────────────────────────────────────────────
# Recommended: "class_decoder_source_bn_aug_reg" (production default, Rung 7)
# For ablation studies, change this to any name from ExperimentConfigs.

MODEL_CONFIG = "class_decoder_source_bn_aug_reg"   # ← Production default
Z_DIM        = 20     # Latent dimension. 10–32; larger = richer but needs more data.
BETA         = 0.9    # KL weight in ELBO. Higher = more structured, Gaussian-like latent space.

# ─── Modality weights ─────────────────────────────────────────────────────────
# Relative contribution to reconstruction loss. All 1.0 means equal weighting.
WAVE_WEIGHT = 1.0
ISI_WEIGHT  = 1.0
ACG_WEIGHT  = 1.0

# ─── Training hyperparameters ─────────────────────────────────────────────────
LEARNING_RATE           = 1e-3   # AdamW learning rate
WEIGHT_DECAY            = 1e-2   # AdamW weight decay
BATCH_SIZE              = 512    # Phase 1 + Phase 2 batch size
SUPERVISED_BATCH_SIZE   = 64     # Phase 3 batch size (smaller → more gradient noise, helps generalization)
GRADIENT_CLIP_VAL       = 1.0
EARLY_STOPPING_PATIENCE = 20     # Epochs without val_loss improvement before stopping

# Epoch limits (production: 100 / 20 / 20; reduce for quick tests)
PRETRAIN_MAX_EPOCHS    = 100
FINETUNE_MAX_EPOCHS    = 20
SUPERVISED_MAX_EPOCHS  = 20

TRAIN_VAL_SPLIT  = 0.8   # Fraction for training (rest for validation)
FINETUNE_SPLIT   = 0.8
FINETUNE_WITHOUT_LABELS = True   # Always recommended
USE_BALANCED_SAMPLING   = True   # Oversample minority classes during supervised training

# ─── Optional Weights & Biases logging ────────────────────────────────────────
USE_WANDB     = False    # Set True if wandb is configured
WANDB_PROJECT = "HIPPIE"

# ─── Computed values ──────────────────────────────────────────────────────────
# Modality output sizes (time points after resampling)
modalities = {"wave": 50, "isi": 100, "acg": 100}
modality_weights = {"wave": WAVE_WEIGHT, "isi": ISI_WEIGHT, "acg": ACG_WEIGHT}
num_sources = max(AVAILABLE_DATASETS.values()) + 1
accelerator = "gpu" if torch.cuda.is_available() else "cpu"

print(f"Config:  {MODEL_CONFIG}  |  z_dim={Z_DIM}  |  beta={BETA}")
print(f"Train:   {TRAINING_DATASET}")
print(f"Predict: {PREDICT_DATASET}")
print(f"Device:  {accelerator.upper()}")

---
## 3. Inspect Configurations

Run this to see a full comparison table of all available configurations.

In [ ]:
config_names = [
    "baseline", "with_source", "with_class", "with_both_embeddings",
    "class_decoder_source", "class_decoder_source_bn",
    "class_decoder_source_bn_strong_aug", "class_decoder_source_bn_aug_reg",
    "full_architecture", "full_architecture_heavy_reg", "HIPPIE_contrastive",
]

rows = []
for name in config_names:
    cfg = getattr(ExperimentConfigs, name)()
    rows.append({
        "Config": name + (" ★" if name == MODEL_CONFIG else ""),
        "Source": "✅" if cfg.use_source_embedding else "❌",
        "Class": "✅" if cfg.use_class_embedding else "❌",
        "Enc↔Class": "✅ leaky" if cfg.encoder_uses_class_embedding else "❌ clean",
        "BN": "✅" if cfg.use_batch_norm else "❌",
        "Aug": f"p={cfg.augment_prob:.1f}" if cfg.use_augmentations else "❌",
        "Dropout": f"{cfg.class_embedding_dropout:.1f}" if cfg.class_embedding_dropout > 0 else "❌",
        "Consist": f"λ={cfg.reconstruction_consistency_weight:.2f}" if cfg.reconstruction_consistency_weight > 0 else "❌",
        "Contrast": "✅" if cfg.use_contrastive_loss else "❌",
    })

df = pd.DataFrame(rows).set_index("Config")
print("HIPPIE Configuration Ladder")
print("=" * 95)
print(df.to_string())
print("\n★ = currently selected config")
print("Enc↔Class=❌ clean → decoder-only conditioning (correct). Enc↔Class=✅ leaky → spatial drift.")

---
## 4. Helper Functions

In [ ]:
import re as _re

def load_dataset(name, source_id, data_dir="./datasets_hippie"):
    """Load all modalities and labels for a single dataset."""
    path = f"{data_dir}/{name}"

    def _sanitize(arr, tag):
        """Replace NaN/Inf with column medians."""
        bad = ~np.isfinite(arr).all(axis=1)
        if bad.any():
            print(f"  [sanitize] {tag}: {bad.sum()} bad rows → replaced with median")
            arr[bad] = np.nanmedian(arr[~bad], axis=0)
        return arr

    wf  = _sanitize(pd.read_csv(f"{path}/waveforms.csv").to_numpy(), "wave")
    isi = _sanitize(pd.read_csv(f"{path}/isi_dist.csv").to_numpy(),  "isi")
    acg_path = f"{path}/acg.csv"
    acg = _sanitize(
        pd.read_csv(acg_path).to_numpy() if os.path.exists(acg_path) else np.zeros((len(wf), 100)),
        "acg"
    )

    labels = None
    for fname in ["labels.csv", "celltypes.csv"]:
        lpath = f"{path}/{fname}"
        if os.path.exists(lpath):
            raw = pd.read_csv(lpath).iloc[:, 0].fillna("").to_numpy()
            # Decode bytes strings (legacy HDF5 artefact).
            # Two forms appear in practice:
            #   actual bytes  → b'GoC'  (from direct HDF5 reads)
            #   string repr   → "b'GoC'" (after CSV round-trip of the above)
            decoded = []
            for v in raw:
                if isinstance(v, bytes):
                    decoded.append(v.decode("utf-8"))
                else:
                    s = str(v)
                    m = _re.match(r'^b[\'\"](.*)[\'\"]\s*$', s)
                    decoded.append(m.group(1) if m else s)
            labels = np.array(decoded)
            break

    return wf, isi, acg, labels, source_id


def get_embeddings(loader, lightning_module):
    """Extract encoder z_mean from a DataLoader. Class labels are always masked (test-time behavior)."""
    model = lightning_module.model
    model.eval()
    all_z, all_class_ids = [], []

    device = next(model.parameters()).device
    with torch.no_grad():
        for batch in loader:
            data_dict, labels = batch
            data_dict = {k: v.to(device) for k, v in data_dict.items()}
            labels = labels.to(device)

            source_labels = labels[:, 1] if labels.ndim == 2 else labels

            # Encoder NEVER receives class labels — this is the clean evaluation path
            _, z_mean, _ = model.encode(data_dict, source_labels=source_labels, class_labels=None)

            z = z_mean.detach().cpu().numpy()
            # L2-normalize rows for cosine-distance KNN
            norms = np.linalg.norm(z, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            z = z / norms

            all_z.append(z)
            cls = labels[:, 0].cpu().numpy() if labels.ndim == 2 else labels.cpu().numpy()
            all_class_ids.append(cls)

    return np.concatenate(all_z), np.concatenate(all_class_ids)


def create_balanced_sampler(class_ids):
    """WeightedRandomSampler that gives each class equal expected frequency per batch."""
    unique, counts = np.unique(class_ids, return_counts=True)
    w = 1.0 / counts
    sample_w = np.zeros(len(class_ids))
    for label, weight in zip(unique, w):
        sample_w[class_ids == label] = weight
    return WeightedRandomSampler(torch.FloatTensor(sample_w), len(class_ids), replacement=True)


def make_loader(dataset, batch_size, sampler=None, shuffle=True):
    """DataLoader with none_safe_collate (handles samples that return None on bad data)."""
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=(shuffle and sampler is None),
        collate_fn=none_safe_collate,
        num_workers=0,
        drop_last=True,
    )


def make_trainer(max_epochs, patience=20):
    """Standard trainer with early stopping and no wandb logger."""
    logger = False
    if USE_WANDB:
        import wandb
        logger = pl.loggers.WandbLogger(project=WANDB_PROJECT)
    return pl.Trainer(
        max_epochs=max_epochs,
        accelerator=accelerator,
        callbacks=[
            pl.callbacks.ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min"),
            pl.callbacks.EarlyStopping(monitor="val_loss", patience=patience, mode="min"),
        ],
        gradient_clip_val=GRADIENT_CLIP_VAL,
        enable_progress_bar=True,
        enable_model_summary=False,
        logger=logger,
    )


def plot_training_curves(module, title):
    """Plot epoch-level loss components from a MultiModalCVAETrainModule."""
    if not module.train_epoch_history:
        print("No epoch history — model may not have been trained.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=12)

    te = [h["epoch"] for h in module.train_epoch_history]
    ve = [h["epoch"] for h in module.val_epoch_history]

    axes[0].plot(te, [h["total"] for h in module.train_epoch_history], label="train", lw=1.5)
    if ve:
        axes[0].plot(ve, [h["total"] for h in module.val_epoch_history], label="val", lw=1.5)
    axes[0].set(title="Total Loss", xlabel="Epoch"); axes[0].legend()

    axes[1].plot(te, [h["mse"] for h in module.train_epoch_history], label="MSE (recon)", lw=1.5)
    axes[1].plot(te, [h["kl"]  for h in module.train_epoch_history], label="KL",          lw=1.5)
    if module.use_contrastive_loss:
        axes[1].plot(te, [h["contrastive"] for h in module.train_epoch_history], label="Contrastive", lw=1.5)
    axes[1].set(title="Loss Components", xlabel="Epoch"); axes[1].legend()

    plt.tight_layout(); plt.show()


print("Helper functions defined.")

---
## 5. Load and Visualize the Data

Before training, let's inspect the three modalities to understand what HIPPIE is working with.

In [ ]:
# Load training dataset
train_wf, train_isi, train_acg, train_labels, train_source_id = load_dataset(
    TRAINING_DATASET, AVAILABLE_DATASETS[TRAINING_DATASET]
)

print(f"Dataset:    {TRAINING_DATASET}")
print(f"Waveforms:  {train_wf.shape}  (n_cells × n_timepoints)")
print(f"ISI dists:  {train_isi.shape}  (n_cells × n_bins)")
print(f"ACGs:       {train_acg.shape}  (n_cells × n_bins)")

if train_labels is not None:
    unique_labels, counts = np.unique(train_labels, return_counts=True)
    # Filter empty labels
    valid_mask = unique_labels != ""
    unique_labels, counts = unique_labels[valid_mask], counts[valid_mask]

    print(f"\nClass distribution ({len(unique_labels)} classes):")
    for label, count in zip(unique_labels, counts):
        pct = 100 * count / len(train_labels)
        bar = "█" * int(pct / 2 + 0.5)
        print(f"  {str(label):<20} {count:>5}  {pct:5.1f}%  {bar}")

In [ ]:
# Plot mean ± std of each modality for each cell type
if train_labels is not None:
    labeled_mask = train_labels != ""
    wf_lab  = train_wf[labeled_mask]
    isi_lab = train_isi[labeled_mask]
    acg_lab = train_acg[labeled_mask]
    labs    = train_labels[labeled_mask]

    unique_classes = np.unique(labs)
    n_cls = len(unique_classes)
    colors = plt.cm.tab10(np.linspace(0, 1, n_cls))

    modality_data   = [wf_lab, np.log(isi_lab + 1), acg_lab]
    modality_titles = ["Waveform (spike shape)", "ISI Distribution (log)", "Autocorrelogram"]
    x_labels        = ["Time (samples)", "ISI bin (log-spaced)", "Lag (ms)"]

    fig, axes = plt.subplots(3, n_cls, figsize=(3 * n_cls, 9), sharey="row")
    if n_cls == 1:
        axes = axes[:, np.newaxis]
    fig.suptitle(f"{TRAINING_DATASET}: Mean ± std per cell type", fontsize=12, y=1.01)

    for col, (cls, color) in enumerate(zip(unique_classes, colors)):
        mask = labs == cls
        for row, (data, mtitle, xlabel) in enumerate(zip(modality_data, modality_titles, x_labels)):
            ax = axes[row, col]
            mean = data[mask].mean(axis=0)
            std  = data[mask].std(axis=0)
            x = np.arange(len(mean))
            ax.plot(x, mean, color=color, lw=2)
            ax.fill_between(x, mean - std, mean + std, alpha=0.25, color=color)
            ax.set_title(str(cls), fontsize=9, pad=2)
            ax.set_xlabel(xlabel, fontsize=7)
            if col == 0:
                ax.set_ylabel(mtitle, fontsize=8)
            ax.tick_params(labelsize=6)

    plt.tight_layout()
    plt.show()
    print("Shaded band = ±1 std across all cells of that type.")
    print("Notice how some types are well-separated in waveform but overlap in ISI, and vice versa.")
    print("HIPPIE fuses all three modalities to exploit complementary information.")

---
## 6. Build the Model

Instantiate the selected `CVAEConfig` and print a human-readable summary.

In [ ]:
# Load the selected configuration preset
config = getattr(ExperimentConfigs, MODEL_CONFIG)()

print(f"\nSelected Configuration: {MODEL_CONFIG}")
print("=" * 55)

# Encoder conditioning (what the encoder sees)
print("\nEncoder inputs:")
print(f"  Waveform, ISI, ACG:     always")
print(f"  Source embedding:       {'✅' if config.use_source_embedding else '❌'}")
print(f"  Super region embedding: {'✅' if config.use_super_region_embedding else '❌'}")
print(f"  Layer embedding:        {'✅' if config.use_layer_embedding else '❌'}")
print(f"  Class embedding:        {'✅ (symmetric — LEAKY!)' if config.encoder_uses_class_embedding and config.use_class_embedding else '❌ (encoder is class-agnostic — correct)'}")

# Decoder conditioning (what the decoder sees during training)
print("\nDecoder inputs (training only):")
print(f"  z (latent sample):      always")
print(f"  Source embedding:       {'✅' if config.use_source_embedding else '❌'}")
print(f"  Class embedding:        {'✅' if config.use_class_embedding else '❌'}")
print(f"  Super region embedding: {'✅' if config.use_super_region_embedding else '❌'}")
print(f"  Layer embedding:        {'✅' if config.use_layer_embedding else '❌'}")

# Architecture
print("\nArchitecture:")
print(f"  Fusion encoder:         {'✅' if config.use_fusion_encoder else '❌'}")
print(f"  Batch normalization:    {'✅' if config.use_batch_norm else '❌'}")
print(f"  z_dim:                  {Z_DIM}")
print(f"  β (KL weight):          {config.beta}")

# Augmentation
if config.use_augmentations:
    print("\nAugmentation (pretrain + finetune phases):")
    print(f"  Probability per batch:  {config.augment_prob}")
    print(f"  Gaussian noise σ:       {config.noise_std}")
    print(f"  Amplitude scale range:  {config.amplitude_scale_range}")
    print(f"  Smoothing σ range:      {config.smoothing_sigma_range}")
    print(f"  During supervised:      {'✅' if config.augment_supervised else '❌'}")
else:
    print("\nAugmentation: ❌ disabled")

# Regularization
print("\nDecoder regularization:")
print(f"  Class embedding dropout: {config.class_embedding_dropout}")
print(f"  Consistency loss weight: {config.reconstruction_consistency_weight}")
print(f"  Embedding warmup epochs: {config.embedding_warmup_epochs}")
print(f"  Contrastive loss:        {'✅ (w=' + str(config.contrastive_weight) + ')' if config.use_contrastive_loss else '❌'}")

---
## 7. Phase 1: Pretraining (Unsupervised)

**Goal:** Learn general electrophysiological features from diverse datasets.

**Data:** All datasets *except* training and prediction datasets (they remain held out).

**Labels used:** Only source IDs — no cell type labels. The CVAE reconstructs signals unsupervised.

**Why this helps:** Diverse pretraining data exposes the encoder to a wide variety of waveform shapes and firing patterns. The model learns domain-general compression before any label information is introduced.

In [ ]:
print("PHASE 1: PRETRAINING")
print("-" * 45)

pretrain_datasets = {k: v for k, v in AVAILABLE_DATASETS.items()
                     if k not in (TRAINING_DATASET, PREDICT_DATASET)}

print(f"Pretraining on {len(pretrain_datasets)} datasets (held out: {TRAINING_DATASET}, {PREDICT_DATASET}):")

datasets_pt = []
joint_model = None

for name, sid in pretrain_datasets.items():
    wf, isi, acg, _, _ = load_dataset(name, sid)
    # Label = source ID only (no class labels in pretraining)
    src_labels = np.full(len(wf), sid, dtype=int)
    ds = MultiModalEphysDataset({"wave": wf, "isi": isi, "acg": acg}, src_labels,
                                 mode="multi", modality_sizes=modalities)
    if config.use_augmentations and config.augment_pretraining:
        ds = AugmentedMultiModalEphysDataset(ds, config, phase="pretraining")
    datasets_pt.append(ds)
    print(f"  {name}: {len(wf)} cells")

if not datasets_pt:
    print("No pretraining datasets — skipping Phase 1.")
else:
    all_pt = torch.utils.data.ConcatDataset(datasets_pt)
    n_pt = len(all_pt)
    n_pt_train, n_pt_val = int(TRAIN_VAL_SPLIT * n_pt), n_pt - int(TRAIN_VAL_SPLIT * n_pt)
    pt_train, pt_val = random_split(all_pt, [n_pt_train, n_pt_val])

    loader_pt_train = make_loader(pt_train, BATCH_SIZE)
    loader_pt_val   = make_loader(pt_val,   BATCH_SIZE, shuffle=False)

    print(f"\nTotal: {n_pt} cells  ({n_pt_train} train / {n_pt_val} val)")
    print(f"Batch size: {BATCH_SIZE}  →  {len(loader_pt_train)} train batches / epoch")

In [ ]:
if datasets_pt:
    base_model_pt = MultiModalCVAE(
        modalities=modalities, z_dim=Z_DIM, config=config,
        num_sources=num_sources,
        num_classes=10,   # Placeholder — class embedding not used in encoder
    )
    train_module_pt = MultiModalCVAETrainModule(
        base_model_pt, config=config, modality_weights=modality_weights,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )

    n_params = sum(p.numel() for p in base_model_pt.parameters() if p.requires_grad)
    print(f"Model parameters: {n_params:,}")
    print(f"Starting pretraining (max {PRETRAIN_MAX_EPOCHS} epochs, early stop patience={EARLY_STOPPING_PATIENCE})...")

    trainer_pt = make_trainer(PRETRAIN_MAX_EPOCHS, EARLY_STOPPING_PATIENCE)
    trainer_pt.fit(train_module_pt, loader_pt_train, loader_pt_val)

    joint_model = base_model_pt
    print(f"\nPretraining stopped at epoch {trainer_pt.current_epoch}.")
    plot_training_curves(train_module_pt, title="Phase 1: Pretraining")

---
## 8. Phase 2: Fine-tuning on Target Dataset (Unsupervised)

**Goal:** Adapt the pretrained representation to the target dataset's recording conditions.

**Data:** The training dataset (no labels — only source ID).

**Why this helps:** Pretraining biases the model toward the pretraining datasets. Fine-tuning shifts the latent space to better match the target recording conditions, reducing domain gap before supervised training.

If Phase 1 was skipped, this cell trains from scratch.

In [ ]:
print("PHASE 2: FINE-TUNING")
print("-" * 45)
print(f"Fine-tuning on: {TRAINING_DATASET}")
print(f"Labels used:    source ID only (no class labels)")

if train_labels is None:
    raise ValueError(f"{TRAINING_DATASET} has no labels — it can't be used as the training dataset.")

if FINETUNE_WITHOUT_LABELS and joint_model is not None:
    ft_src = np.full(len(train_wf), train_source_id, dtype=int)
    ft_ds  = MultiModalEphysDataset(
        {"wave": train_wf, "isi": train_isi, "acg": train_acg},
        ft_src, mode="multi", modality_sizes=modalities
    )
    if config.use_augmentations and config.augment_finetuning:
        ft_ds = AugmentedMultiModalEphysDataset(ft_ds, config, phase="finetuning")

    n_ft = len(ft_ds)
    n_ft_train = int(FINETUNE_SPLIT * n_ft)
    ft_train, ft_val = random_split(ft_ds, [n_ft_train, n_ft - n_ft_train])

    loader_ft_train = make_loader(ft_train, BATCH_SIZE)
    loader_ft_val   = make_loader(ft_val,   BATCH_SIZE, shuffle=False)

    train_module_ft = MultiModalCVAETrainModule(
        joint_model, config=config, modality_weights=modality_weights,
        learning_rate=LEARNING_RATE / 10,  # Lower LR for fine-tuning
        weight_decay=WEIGHT_DECAY,
    )
    print(f"\nFine-tuning {n_ft_train} cells (max {FINETUNE_MAX_EPOCHS} epochs)...")
    trainer_ft = make_trainer(FINETUNE_MAX_EPOCHS, EARLY_STOPPING_PATIENCE)
    trainer_ft.fit(train_module_ft, loader_ft_train, loader_ft_val)
    print("Fine-tuning complete.")
    plot_training_curves(train_module_ft, title="Phase 2: Fine-tuning")
else:
    print("Skipping fine-tuning (no pretrained model available or FINETUNE_WITHOUT_LABELS=False).")

---
## 9. Phase 3: Supervised Training

**Goal:** Refine the latent space using ground-truth cell type labels.

**What's new compared to Phase 2:**
- Labels format is now `[class_id, source_id]` (2-column)
- The **decoder** receives class embeddings — but the encoder still doesn't!
- Reconstruction consistency loss and class embedding dropout activate

**Why the class embedding is only in the decoder:**
The class embedding guides reconstruction during training, teaching the latent `z` to encode class-discriminative features. But since the encoder never sees class labels, `z` must encode this discriminability intrinsically — making the evaluation path perfectly clean and matching real-world use.

In [ ]:
print("PHASE 3: SUPERVISED TRAINING")
print("-" * 45)

# Filter out unlabeled cells (empty string labels)
labeled_mask = train_labels != ""
wf_labeled  = train_wf[labeled_mask]
isi_labeled = train_isi[labeled_mask]
acg_labeled = train_acg[labeled_mask]
labels_labeled = train_labels[labeled_mask]

# Encode string labels to contiguous integers
le = LabelEncoder().fit(labels_labeled)
labels_enc = le.transform(labels_labeled)
num_class_labels = len(le.classes_)

print("Label encoding:")
for i, cls in enumerate(le.classes_):
    n = (labels_enc == i).sum()
    print(f"  {i}: {str(cls):<20} ({n} cells)")

# Train / val split (stratified)
idx = np.arange(len(wf_labeled))
try:
    tr_idx, val_idx = train_test_split(idx, test_size=1-TRAIN_VAL_SPLIT,
                                       stratify=labels_enc, random_state=42)
except ValueError:
    tr_idx, val_idx = train_test_split(idx, test_size=1-TRAIN_VAL_SPLIT, random_state=42)

# Labels format for the model: [class_id, source_id]
def make_labels_2col(cls_ids, src_id):
    return np.stack([cls_ids, np.full(len(cls_ids), src_id, dtype=int)], axis=1)

labels_tr  = make_labels_2col(labels_enc[tr_idx],  train_source_id)
labels_val = make_labels_2col(labels_enc[val_idx],  train_source_id)

ds_tr  = MultiModalEphysDataset(
    {"wave": wf_labeled[tr_idx],  "isi": isi_labeled[tr_idx],  "acg": acg_labeled[tr_idx]},
    labels_tr,  mode="multi", modality_sizes=modalities
)
ds_val = MultiModalEphysDataset(
    {"wave": wf_labeled[val_idx], "isi": isi_labeled[val_idx], "acg": acg_labeled[val_idx]},
    labels_val, mode="multi", modality_sizes=modalities
)

sampler = create_balanced_sampler(labels_enc[tr_idx]) if USE_BALANCED_SAMPLING else None
loader_sup_tr  = make_loader(ds_tr,  SUPERVISED_BATCH_SIZE, sampler=sampler)
loader_sup_val = make_loader(ds_val, SUPERVISED_BATCH_SIZE, shuffle=False)

print(f"\nTrain: {len(ds_tr)} cells  |  Val: {len(ds_val)} cells")
print(f"Balanced sampling: {'✅' if USE_BALANCED_SAMPLING else '❌'}")

In [ ]:
# Build or reuse model
if joint_model is None:
    # No pretrained model — initialize fresh
    joint_model = MultiModalCVAE(
        modalities=modalities, z_dim=Z_DIM, config=config,
        num_sources=num_sources, num_classes=num_class_labels,
    )
    print("Initializing fresh model (no pretrained weights).")
else:
    # Reinitialize class embedding for the actual number of classes
    if joint_model.class_embedding is not None:
        joint_model.class_embedding = nn.Embedding(num_class_labels, joint_model.class_hidden_dim)
    print(f"Reusing encoder weights from pretraining. Class embedding reinitialized for {num_class_labels} classes.")

train_module_sup = MultiModalCVAETrainModule(
    joint_model, config=config, modality_weights=modality_weights,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)

print(f"\nStarting supervised training (max {SUPERVISED_MAX_EPOCHS} epochs)...")
print(f"  Encoder sees class labels: {'YES (leaky)' if config.encoder_uses_class_embedding else 'NO (clean)'}")
print(f"  Decoder class embedding:   {'YES' if config.use_class_embedding else 'NO'}")
print(f"  Consistency loss active:   {'YES (λ=' + str(config.reconstruction_consistency_weight) + ')' if config.reconstruction_consistency_weight > 0 else 'NO'}")

trainer_sup = make_trainer(SUPERVISED_MAX_EPOCHS, EARLY_STOPPING_PATIENCE)
trainer_sup.fit(train_module_sup, loader_sup_tr, loader_sup_val)

print(f"\nSupervised training complete (stopped at epoch {trainer_sup.current_epoch}).")
plot_training_curves(train_module_sup, title="Phase 3: Supervised Training")

---
## 10. Evaluation

**How evaluation works:**
1. Extract `z_mean` from the encoder on training data — **class labels are NOT passed** (test-time behavior)
2. Fit a KNN classifier on these embeddings using true labels
3. Extract embeddings from the prediction dataset (same clean encoder pass)
4. Predict cell types via KNN
5. Compute balanced accuracy (corrects for class imbalance)

**Note:** The encoder receives only waveform/ISI/ACG and source embedding — no class embedding at all. This matches real-world usage exactly.

In [ ]:
print("EVALUATION")
print("-" * 45)

# Extract training embeddings (all labeled cells, class labels masked from encoder)
labels_all = make_labels_2col(labels_enc, train_source_id)
ds_all = MultiModalEphysDataset(
    {"wave": wf_labeled, "isi": isi_labeled, "acg": acg_labeled},
    labels_all, mode="multi", modality_sizes=modalities
)
loader_all = make_loader(ds_all, BATCH_SIZE, shuffle=False)

print("Extracting training embeddings (class labels masked — simulating test time)...")
train_z, train_cls_ids = get_embeddings(loader_all, train_module_sup)
train_labels_str = le.inverse_transform(train_cls_ids.astype(int))
print(f"  → {train_z.shape[0]} embeddings, z_dim={train_z.shape[1]}")

# Load prediction dataset
print(f"\nLoading prediction dataset: {PREDICT_DATASET}...")
pred_wf, pred_isi, pred_acg, pred_labels, pred_source_id = load_dataset(
    PREDICT_DATASET, AVAILABLE_DATASETS[PREDICT_DATASET]
)

# Handle missing labels
has_pred_labels = pred_labels is not None and len(pred_labels) > 0
if has_pred_labels:
    labeled_pred_mask = pred_labels != ""
    print(f"  {labeled_pred_mask.sum()} / {len(pred_labels)} cells have labels")

# Build prediction dataset (class ID = -1 for unlabeled; encoder will ignore it)
pred_cls_dummy = np.full(len(pred_wf), -1, dtype=int)
pred_labels_2col = make_labels_2col(pred_cls_dummy, pred_source_id)
ds_pred = MultiModalEphysDataset(
    {"wave": pred_wf, "isi": pred_isi, "acg": pred_acg},
    pred_labels_2col, mode="multi", modality_sizes=modalities
)
loader_pred = make_loader(ds_pred, BATCH_SIZE, shuffle=False)

print("Extracting prediction embeddings...")
pred_z, _ = get_embeddings(loader_pred, train_module_sup)
print(f"  → {pred_z.shape[0]} embeddings")

In [ ]:
# KNN: cross-validate to select K, then predict
print("Fitting KNN classifier (sweeping K values)...")

k_range = [3, 5, 7, 11, 15, 21]
k_range = [k for k in k_range if k < len(train_z)]

z_tr_knn, z_val_knn, y_tr_knn, y_val_knn = train_test_split(
    train_z, train_labels_str,
    test_size=0.2, stratify=train_labels_str, random_state=42
)

print(f"  {'K':>4}   Val Balanced Accuracy")
print(f"  {'-'*30}")
best_k, best_knn, best_score = None, None, -1
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(z_tr_knn, y_tr_knn)
    score = balanced_accuracy_score(y_val_knn, knn.predict(z_val_knn))
    marker = " ← best" if score > best_score else ""
    print(f"  {k:>4}   {score:.3f}{marker}")
    if score > best_score:
        best_score, best_k = score, k

# Refit on all training data
best_knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine", n_jobs=-1)
best_knn.fit(train_z, train_labels_str)
predictions = best_knn.predict(pred_z)

print(f"\nSelected K={best_k}  (val balanced accuracy: {best_score:.3f})")

In [ ]:
# Report accuracy if ground-truth labels are available
if has_pred_labels and labeled_pred_mask.sum() > 0:
    true_cls = pred_labels[labeled_pred_mask]
    pred_cls = predictions[labeled_pred_mask]

    # Only evaluate on classes the KNN has seen
    shared_mask = np.isin(true_cls, best_knn.classes_)
    true_eval = true_cls[shared_mask]
    pred_eval = pred_cls[shared_mask]

    if len(true_eval) > 0:
        ba  = balanced_accuracy_score(true_eval, pred_eval)
        acc = (true_eval == pred_eval).mean()

        print(f"\n{'=' * 50}")
        print(f"  RESULTS: {TRAINING_DATASET} → {PREDICT_DATASET}")
        print(f"  Config:            {MODEL_CONFIG}")
        print(f"  Balanced Accuracy: {ba:.4f}")
        print(f"  Raw Accuracy:      {acc:.4f}")
        print(f"  Cells evaluated:   {len(true_eval)} / {len(pred_labels)}")
        print(f"{'=' * 50}")

        print("\nPer-class accuracy (recall):")
        for cls in sorted(set(true_eval)):
            mask = true_eval == cls
            recall = (pred_eval[mask] == cls).mean()
            bar = "█" * int(recall * 25)
            print(f"  {str(cls):<20} {recall:.3f}  {bar}")
    else:
        print("No overlapping classes between training and prediction datasets.")
else:
    print(f"Predictions made for {len(predictions)} cells.")
    print("Prediction distribution:")
    for cls, cnt in zip(*np.unique(predictions, return_counts=True)):
        print(f"  {str(cls):<20} {cnt}")

---
## 11. Visualization

**t-SNE** projects the `z_dim`-dimensional latent space to 2D. Well-separated clusters = the model is encoding class-discriminative features.

In [ ]:
print("Running t-SNE (may take 30–90 seconds)...")
n_both = len(train_z) + len(pred_z)
all_z = np.vstack([train_z, pred_z])

perp = min(30, n_both // 4)
tsne = TSNE(n_components=2, random_state=42, perplexity=perp, n_iter=1000, learning_rate="auto", init="pca")
all_2d = tsne.fit_transform(all_z)

train_2d = all_2d[:len(train_z)]
pred_2d  = all_2d[len(train_z):]
print(f"t-SNE complete (perplexity={perp}).")

In [ ]:
all_classes = sorted(set(list(train_labels_str) + list(predictions)))
cmap = plt.cm.tab10
c2color = {cls: cmap(i / max(len(all_classes) - 1, 1)) for i, cls in enumerate(all_classes)}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Latent Space — {MODEL_CONFIG}  (z_dim={Z_DIM})", fontsize=13)

# Left: training set colored by true labels
ax = axes[0]
for cls in all_classes:
    m = train_labels_str == cls
    if m.any():
        ax.scatter(train_2d[m, 0], train_2d[m, 1], c=[c2color[cls]], label=str(cls),
                   s=12, alpha=0.75, edgecolors="none")
ax.set_title(f"Training: {TRAINING_DATASET}\n(true labels)", fontsize=10)
ax.legend(markerscale=2.5, fontsize=8)
ax.axis("off")

# Right: prediction set colored by KNN-predicted labels
ax = axes[1]
for cls in all_classes:
    m = predictions == cls
    if m.any():
        ax.scatter(pred_2d[m, 0], pred_2d[m, 1], c=[c2color[cls]], label=str(cls),
                   s=12, alpha=0.75, edgecolors="none")
ax.set_title(f"Prediction: {PREDICT_DATASET}\n(KNN predicted labels)", fontsize=10)
ax.legend(markerscale=2.5, fontsize=8)
ax.axis("off")

plt.tight_layout()
plt.show()
print("Tight clusters with consistent colors = class-discriminative latent space.")

In [ ]:
# Confusion matrix (if ground truth available)
if has_pred_labels and labeled_pred_mask.sum() > 0 and len(true_eval) > 0:
    shared_cls = sorted(set(true_eval) & set(pred_eval))
    cm = confusion_matrix(true_eval, pred_eval, labels=shared_cls, normalize="true")

    fig, ax = plt.subplots(figsize=(max(6, len(shared_cls)), max(5, len(shared_cls) - 1)))
    disp = ConfusionMatrixDisplay(cm, display_labels=[str(c) for c in shared_cls])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(f"Confusion Matrix (row-normalized recall)\n{TRAINING_DATASET} → {PREDICT_DATASET}")
    plt.xticks(rotation=35, ha="right", fontsize=8)
    plt.tight_layout()
    plt.show()
    print("Rows = true class, Columns = predicted class. Diagonal = recall per class.")
    print("Ideal = identity matrix (all 1s on diagonal).")

---
## 12. Ablation: Compare Conditioning Configurations

Run multiple configurations and compare their balanced accuracy. This reproduces the conditioning ladder from Figure 2.

**Warning:** Each config trains from scratch. Reduce `N_EPOCHS` for quick comparisons.

In [ ]:
def run_one_config(config_name, n_epochs=15):
    """Train supervised-only and return balanced accuracy on the prediction dataset."""
    cfg = getattr(ExperimentConfigs, config_name)()

    model = MultiModalCVAE(
        modalities=modalities, z_dim=Z_DIM, config=cfg,
        num_sources=num_sources, num_classes=num_class_labels,
    )
    tm = MultiModalCVAETrainModule(
        model, config=cfg, modality_weights=modality_weights,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    trainer = pl.Trainer(
        max_epochs=n_epochs, accelerator=accelerator,
        enable_progress_bar=False, enable_model_summary=False, logger=False,
        callbacks=[pl.callbacks.EarlyStopping(monitor="val_loss", patience=5, mode="min")],
    )
    trainer.fit(tm, loader_sup_tr, loader_sup_val)

    z_tr, y_tr = get_embeddings(loader_all, tm)
    z_pr, _    = get_embeddings(loader_pred, tm)

    knn = KNeighborsClassifier(n_neighbors=7, metric="cosine").fit(
        z_tr, le.inverse_transform(y_tr.astype(int))
    )
    preds = knn.predict(z_pr)

    if has_pred_labels and labeled_pred_mask.sum() > 0:
        t = pred_labels[labeled_pred_mask]
        p = preds[labeled_pred_mask]
        shared = np.isin(t, knn.classes_)
        if shared.any():
            return balanced_accuracy_score(t[shared], p[shared])
    return None


# Configuration ladder to compare
ABLATION_LADDER = [
    "baseline",                           # Rung 0: Pure VAE
    "with_source",                        # Rung 1: + source embedding
    "with_both_embeddings",               # Rung 3: symmetric (leaky — expect poor results)
    "class_decoder_source_bn",            # Rung 5: asymmetric + BN
    "class_decoder_source_bn_aug_reg",    # Rung 7: PRODUCTION
]
N_EPOCHS = 15  # Reduce for faster testing

print(f"Running conditioning ladder ({N_EPOCHS} epochs each, supervised only):")
print(f"  {'Config':<40} {'Bal. Acc.':>12}")
print(f"  {'-' * 54}")

ablation_results = {}
for cfg_name in ABLATION_LADDER:
    ba = run_one_config(cfg_name, n_epochs=N_EPOCHS)
    ablation_results[cfg_name] = ba
    if ba is not None:
        bar = "█" * int(ba * 30)
        star = " ★" if cfg_name == "class_decoder_source_bn_aug_reg" else ""
        is_leaky = cfg_name in ("with_both_embeddings", "with_class", "full_architecture")
        note = " [leaky!]" if is_leaky else ""
        print(f"  {cfg_name:<40} {ba:.3f}  {bar}{star}{note}")
    else:
        print(f"  {cfg_name:<40} N/A")

print("\n★ = production config  |  [leaky!] = symmetric encoder (spatial drift failure mode)")

In [ ]:
# Plot ablation results
valid = {k: v for k, v in ablation_results.items() if v is not None}
if valid:
    leaky_configs = {"with_both_embeddings", "with_class"}
    colors = ["coral" if k in leaky_configs else
              "darkorange" if k == "class_decoder_source_bn_aug_reg" else
              "steelblue" for k in valid]

    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.barh(list(valid.keys()), list(valid.values()), color=colors, alpha=0.85)

    ax.axvline(1 / num_class_labels, color="red", ls="--", lw=1.2,
               label=f"Chance (1/{num_class_labels}={1/num_class_labels:.2f})")
    ax.set_xlabel("Balanced Accuracy")
    ax.set_title(f"Conditioning Ladder: {TRAINING_DATASET} → {PREDICT_DATASET}")
    ax.set_xlim(0, 1.05)
    ax.legend(fontsize=9)

    for bar, val in zip(bars, valid.values()):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=9)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="coral",       label="Symmetric (leaky — spatial drift)"),
        Patch(color="steelblue",   label="Asymmetric (clean)"),
        Patch(color="darkorange",  label="Production (Rung 7)"),
    ] + [plt.Line2D([0], [0], color="red", ls="--", label=f"Chance={1/num_class_labels:.2f}")],
        fontsize=9, loc="lower right")

    plt.tight_layout()
    plt.show()

---
## 13. Advanced: HIPPIE_contrastive

The `HIPPIE_contrastive` config adds a **supervised contrastive loss** on top of the production config.

**How the contrastive loss works:**
- Labeled cells in a batch are *anchors*
- Same-class labeled cells are *positives*; all other cells are *negatives* (including unlabeled)
- Unlabeled cells appear only as negatives — they get pushed away from all labeled clusters
- Temperature-scaled cosine similarity (τ=0.5 default)
- Loss = 0 during pretraining/finetuning (no labels available); activates only in supervised phase

**When to use:** Datasets with many confusable cell types, or when the base config gives weak clustering in t-SNE.

**Caveat:** On small datasets (< 10 cells per class), positive pairs may be sparse, which can destabilize the loss.


In [ ]:
cfg_c = ExperimentConfigs.HIPPIE_contrastive()

print("HIPPIE_contrastive configuration:")
print(f"  Base:                  class_decoder_source_bn_aug_reg (Rung 7)")
print(f"  Contrastive loss:      {'✅' if cfg_c.use_contrastive_loss else '❌'}")
print(f"  Contrastive weight:    {cfg_c.contrastive_weight}")
print(f"  Temperature (τ):       {cfg_c.contrastive_temperature}")
print(f"  Encoder class-agnostic: {not cfg_c.encoder_uses_class_embedding}")
print()
print("To use: set MODEL_CONFIG = 'HIPPIE_contrastive' in Section 2 and re-run.")
print()
print("Total loss during supervised phase:")
print("  L = L_recon + β·L_KL + warmup·λ_consist·L_consistency + λ_contrast·L_contrastive")
print(f"    = L_recon + {cfg_c.beta}·L_KL + warmup·{cfg_c.reconstruction_consistency_weight}·L_consistency + {cfg_c.contrastive_weight}·L_contrastive")

---
## Summary and Tuning Guide

### What you ran

| Phase | Input | Labels used | Goal |
|-------|-------|-------------|------|
| 1. Pretrain | All other datasets | Source IDs only | Domain-general representation |
| 2. Finetune | Target dataset | Source IDs only | Target-domain adaptation |
| 3. Supervised | Target dataset | Class + source IDs | Class-discriminative z |
| 4. Eval | KNN on z (class masked) | None | Balanced accuracy |

### Conditioning types recap

| Embedding | Where | When available | Controls |
|-----------|-------|---------------|----------|
| **Source** | Encoder + Decoder | Always | Technology / dataset bias |
| **Class** | Decoder ONLY (asymmetric) | Train only | Cell-type discriminability |
| **Super region** | Encoder + Decoder | Always | Brain-region baseline variation |
| **Layer** | Encoder + Decoder | Always | Laminar position variation |

### Troubleshooting

| Problem | Likely cause | Try |
|---------|-------------|-----|
| Low accuracy overall | Small z_dim or insufficient training | Increase Z_DIM (15→25→32) |
| Train >> val loss | Overfitting | More augmentation, smaller SUPERVISED_MAX_EPOCHS |
| Chance-level accuracy | Spatial drift | Check `encoder_uses_class_embedding=False` |
| Poor cross-dataset transfer | Domain gap | Add more pretraining datasets |
| Minority classes always wrong | Class imbalance | USE_BALANCED_SAMPLING=True (already default) |
| t-SNE clusters mixed | Weak class signal | Try `HIPPIE_contrastive` |

### Production CLI (for large-scale benchmarking)

```bash
# 5-fold cross-validation on a single dataset
python cross_dataset_script.py \
    --config class_decoder_source_bn_aug_reg \
    --dataset hausser_cell_type \
    --z_dim 20 --beta 0.9 --cv_fold 0

# Cross-dataset transfer
python cross_dataset_script.py \
    --config class_decoder_source_bn_aug_reg \
    --dataset hausser_cell_type \
    --predict-dataset lissberger_labeled_cell_type \
    --z_dim 20 --beta 0.9
```
